# Advanced Analytics + Risk Metrics

Task 6 — Bluestock Mutual Fund Analytics

This notebook computes historical VaR/CVaR, rolling 90-day Sharpe, investor cohorts, SIP continuity, a rule-based recommender, sector HHI, and five advanced insights.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Resolve project root from common notebook/project locations.
HERE = Path.cwd()
ROOT = HERE
for candidate in [HERE, HERE.parent, HERE.parent.parent, Path(r"C:/Users/drdee/Desktop/BlueStock_fintech/capstone_project")]:
    if (candidate / "data").exists():
        ROOT = candidate
        break

PROCESSED = ROOT / "data" / "processed"
RAW = ROOT / "data" / "raw"
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

def load(identifier):
    matches = list(PROCESSED.glob(f"*{identifier}*.csv")) + list(RAW.glob(f"*{identifier}*.csv"))
    if not matches:
        raise FileNotFoundError(f"Could not find dataset containing {identifier}")
    return pd.read_csv(matches[0])

fund = load("01_fund_master")
nav = load("02_nav_history")
perf = load("07_scheme_performance")
tx = load("08_investor_transactions")
hold = load("09_portfolio_holdings")
category_inflows = load("05_category_inflows")

category_inflows["month"] = pd.to_datetime(category_inflows["month"])

nav["date"] = pd.to_datetime(nav["date"])
tx["transaction_date"] = pd.to_datetime(tx["transaction_date"])
hold["portfolio_date"] = pd.to_datetime(hold["portfolio_date"])
nav = nav.sort_values(["amfi_code", "date"])
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()
name_map = fund.set_index("amfi_code")["scheme_name"].to_dict()
print("Project root:", ROOT)
print("Funds:", len(fund), "NAV rows:", len(nav), "Transactions:", len(tx))

## 1. Historical VaR and CVaR

Historical 95% VaR is the 5th percentile of daily returns. CVaR is the mean of returns at or below that VaR threshold.

In [ ]:
var_rows=[]
for code, g in nav.groupby("amfi_code"):
    returns = g["daily_return"].dropna()
    if returns.empty: continue
    var95 = returns.quantile(0.05)
    cvar95 = returns[returns <= var95].mean()
    var_rows.append({"amfi_code": code, "fund_name": name_map.get(code, ""), "var_95": var95, "cvar_95": cvar95, "observations": len(returns)})
var_cvar = pd.DataFrame(var_rows).sort_values("var_95")
var_cvar.to_csv(OUTPUTS / "var_cvar_report.csv", index=False)
var_cvar.head(10)

## 2. Rolling 90-Day Sharpe Ratio

The rolling Sharpe ratio is annualised using √252.

In [ ]:
key_codes = list(fund["amfi_code"].head(5))
plt.figure(figsize=(12,7))
for code in key_codes:
    r = nav.loc[nav.amfi_code.eq(code)].set_index("date")["daily_return"]
    rolling_sharpe = r.rolling(90).mean() / r.rolling(90).std() * np.sqrt(252)
    plt.plot(rolling_sharpe.index, rolling_sharpe, label=name_map.get(code, str(code)))
plt.title("Rolling 90-Day Sharpe Ratio — Five Key Funds")
plt.xlabel("Date"); plt.ylabel("Annualised Rolling Sharpe Ratio")
plt.legend(fontsize=7); plt.grid(alpha=0.25); plt.tight_layout()
plt.savefig(OUTPUTS / "rolling_sharpe_chart.png", dpi=180)
plt.show()

## 3. Investor Cohort Analysis

In [ ]:
first_year = tx.groupby("investor_id")["transaction_date"].min().dt.year.rename("cohort_year")
txc = tx.join(first_year, on="investor_id")
sip = txc[txc["transaction_type"].str.strip().str.lower().eq("sip")].copy()
cohort = txc.groupby("cohort_year").agg(investors=("investor_id","nunique"), total_invested_inr=("amount_inr","sum"), transactions=("investor_id","size")).reset_index()
avg_sip = sip.groupby("cohort_year")["amount_inr"].mean().rename("avg_sip_amount_inr")
cohort = cohort.merge(avg_sip, on="cohort_year", how="left")
fund_pref = (txc.groupby(["cohort_year","amfi_code"]).size().reset_index(name="transactions")
             .sort_values(["cohort_year","transactions"], ascending=[True,False])
             .drop_duplicates("cohort_year"))
fund_pref["top_fund"] = fund_pref["amfi_code"].map(name_map)
cohort = cohort.merge(fund_pref[["cohort_year","top_fund"]], on="cohort_year", how="left")
cohort

## 4. SIP Continuity Analysis

In [ ]:
sip = sip.sort_values(["investor_id","transaction_date"])
sip_counts = sip.groupby("investor_id").size()
eligible = sip_counts[sip_counts >= 6].index
el = sip[sip.investor_id.isin(eligible)].copy()
el["gap_days"] = el.groupby("investor_id")["transaction_date"].diff().dt.days
continuity = el.groupby("investor_id").agg(sip_transactions=("transaction_date","size"), avg_gap_days=("gap_days","mean")).reset_index()
continuity["status"] = np.where(continuity["avg_gap_days"] > 35, "At-Risk", "Regular")
continuity_rate = (continuity["status"].eq("Regular").mean() * 100) if len(continuity) else np.nan
print(f"Eligible investors: {len(continuity)}")
print(f"SIP continuity rate: {continuity_rate:.2f}%")
continuity.head(10)

## 5. Simple Risk-Based Fund Recommender

In [ ]:
def recommend_funds(risk_appetite, n=3):
    mapping = {"low":"Low", "moderate":"Moderate", "high":"High"}
    grade = mapping.get(str(risk_appetite).strip().lower())
    if grade is None:
        raise ValueError("risk_appetite must be Low, Moderate, or High")
    cols=["amfi_code","scheme_name","risk_grade","sharpe_ratio","return_1yr_pct","expense_ratio_pct"]
    out = perf.loc[perf["risk_grade"].eq(grade), cols].copy()
    return out.sort_values("sharpe_ratio", ascending=False).head(n).reset_index(drop=True)

print("Example — Moderate")
recommend_funds("Moderate")

## 6. Sector HHI Concentration

In [ ]:
h = hold.copy()
h["weight_decimal"] = h["weight_pct"] / 100
sector_weights = h.groupby(["amfi_code","sector"], as_index=False)["weight_decimal"].sum()
hhi = sector_weights.groupby("amfi_code")["weight_decimal"].apply(lambda x: (x**2).sum()).reset_index(name="hhi")
hhi["fund_name"] = hhi["amfi_code"].map(name_map)
equity_codes = set(fund.loc[fund["category"].astype(str).str.contains("Equity", case=False, na=False), "amfi_code"])
hhi_equity = hhi[hhi["amfi_code"].isin(equity_codes)].sort_values("hhi", ascending=False)
hhi_equity

## 7. Five Advanced Insights

1. **Downside risk:** The most negative historical VaR identifies schemes with the lowest 5th-percentile daily return threshold; CVaR shows the average loss in that tail.
2. **Tail severity:** Comparing VaR and CVaR distinguishes a fund with a modest threshold loss from one whose extreme-loss tail is materially deeper.
3. **Investor cohorts:** Cohort totals and average SIP amounts show how investment behaviour differs by the year investors first appeared in the dataset.
4. **SIP continuity:** Investors with six or more SIPs are segmented by average contribution gap; gaps above 35 days are flagged as at-risk under the project rule.
5. **Portfolio concentration:** Equity-fund HHI highlights differences in sector concentration; higher HHI means a larger share of exposure is concentrated in fewer sectors.

## 7A. Category Inflow Analysis

The supplied `05_category_inflows.csv` is included as an additional analytical input. It is used here to provide category-level context for investor flows and to support an additional advanced insight.

In [ ]:
category_summary = (category_inflows.groupby("category", as_index=False)["net_inflow_crore"]
                   .sum().sort_values("net_inflow_crore", ascending=False))
display(category_summary)
top_category = category_summary.iloc[0]
print(f"Highest cumulative category inflow: {top_category['category']} — ₹{top_category['net_inflow_crore']:,.0f} crore")

### Additional insight — Category flows
Category-level net inflows provide a complementary view of investor demand. The category with the largest cumulative net inflow over the supplied period is reported directly from the dataset rather than inferred from fund-level risk metrics.

In [ ]:
print("Top 5 most negative VaR:")
display(var_cvar.head(5))
print("\nLargest investor cohorts by total invested:")
display(cohort.sort_values("total_invested_inr", ascending=False).head())
print("\nHighest HHI equity funds:")
display(hhi_equity.head(5))